# Microbatch Analysis

Analyzes the effect of microbatching on SVD optimizer performance across all datasets.
Microbatching averages groups of per-sample losses before computing the Jacobian,
reducing its row dimension from B to B/mb. At mb=B, the Jacobian is 1×P (standard gradient).

## 1. Setup & Data Loading

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from style import set_style, lr_labels
set_style()

PLOT_DIR = Path('plots/microbatch')
PLOT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Plots will be saved to: {PLOT_DIR.resolve()}")

In [ ]:
# Load results
from style import load_results

SCAN_CONFIGS = {
    #'MNIST (LabelReg)':    'mnist_microbatch_labelreg_scan',
    'Toy 1D':              'toy_1d_microbatch_scan',
    'Polynomial':          'polynomial_microbatch_scan',
}

datasets = {}
for name, scan_name in SCAN_CONFIGS.items():
    try:
        df = load_results(scan_name)
        datasets[name] = df
    except FileNotFoundError:
        print(f"  [skip] {name}: not found")

print(f"\nLoaded {len(datasets)} datasets: {list(datasets.keys())}")


In [ ]:
# Helper functions
def get_loss_curve(row, loss_type='val'):
    return np.array(row['losses'][loss_type])

def get_acc_curve(row, acc_type='val_acc'):
    return np.array(row['losses'].get(acc_type, []))

def sliding_average(data, window=10):
    return np.convolve(data, np.ones(window)/window, mode='valid')

def add_derived_columns(df):
    df = df.copy()
    df['final_val_loss'] = df.apply(lambda r: r['losses']['val'][-1], axis=1)
    df['final_train_loss'] = df.apply(lambda r: r['losses']['train'][-1], axis=1)
    df['final_val_acc'] = df.apply(lambda r: r['losses'].get('val_acc', [np.nan])[-1], axis=1)
    df['final_train_acc'] = df.apply(lambda r: r['losses'].get('train_acc', [np.nan])[-1], axis=1)
    df['total_time'] = df['losses'].apply(lambda l: l.get('total_time', np.nan))
    df['avg_batch_time_train'] = df['losses'].apply(lambda l: l.get('avg_batch_time_train', np.nan))
    # Ensure param_fraction is numeric
    df['param_fraction'] = df['param_fraction'].fillna(1.0).astype(float)
    return df

for name in datasets:
    datasets[name] = add_derived_columns(datasets[name])

for name, df in datasets.items():
    pf_vals = sorted(df['param_fraction'].unique())
    print(f"{name}: {len(df)} runs, param_fractions={pf_vals}")

In [ ]:
mb_sizes_1d = sorted(datasets['Toy 1D']['microbatch_size'].unique())
model_seeds_1d = datasets['Toy 1D']['model_seed'].unique()
k_values_1d = datasets['Toy 1D']['k'].unique()
rtols_1d = datasets['Toy 1D']['rtol'].unique()
lrs_1d = datasets['Toy 1D']['lr'].unique()
print(f"Toy 1D microbatch sizes: {mb_sizes_1d}")
print(f"Toy 1D model_seeds: {model_seeds_1d}")
print(f"Toy 1D k_values: {k_values_1d}")
print(f"Toy 1D learning rates: {lrs_1d}")
print(f"Toy 1D rtols: {rtols_1d}")

print("="*50)

mb_sizes_polynomial = sorted(datasets['Polynomial']['microbatch_size'].unique())
model_seeds_polynomial = datasets['Polynomial']['model_seed'].unique()
k_values_polynomial = datasets['Polynomial']['k'].unique()
rtols_polynomial = datasets['Polynomial']['rtol'].unique()
lrs_polynomial = datasets['Polynomial']['lr'].unique()
print(f"Polynomial microbatch sizes: {mb_sizes_polynomial}")
print(f"Polynomial model_seeds: {model_seeds_polynomial}")
print(f"Polynomial k_values: {k_values_polynomial}")
print(f"Polynomial learning rates: {lrs_polynomial}")
print(f"Polynomial rtols: {rtols_polynomial}")

print("="*50)

#param_fracs_mnist = datasets['MNIST']['param_fraction'].unique()
#model_seeds_mnist = datasets['MNIST']['model_seed'].unique()
#k_values_mnist = datasets['MNIST']['k'].unique()
#rtols_mnist = datasets['MNIST']['rtol'].unique()
#lrs_mnist = datasets['MNIST']['lr'].unique()
#print(f"MNIST param_fractions: {param_fracs_mnist}")
#print(f"MNIST model_seeds: {model_seeds_mnist}")
#print(f"MNIST k_values: {k_values_mnist}")
#print(f"MNIST learning rates: {lrs_mnist}")
#print(f"MNIST rtols: {rtols_mnist}")

# 1D Regression

In [ ]:
colors = sns.color_palette("deep")

fig, ax = plt.subplots(figsize=(8, 6))

K_SEL = 16
RTOL = 1e-3
LR = 0.05

df = datasets['Toy 1D']

for mb_size in mb_sizes_1d:
    losses = []
    for mseed in model_seeds_1d:
        cut = (df['microbatch_size'] == mb_size) & (df['model_seed'] == mseed) & (df['k'] == K_SEL) & (df['rtol'] == RTOL) & (df['lr'] == LR)
        row = df.loc[cut]
        if len(row) > 1:
            print("bad!")
        row = row.iloc[0]
        losses.append(np.array(get_loss_curve(row, 'val')).reshape(1,-1))
    losses = np.concatenate(losses, axis=0)
    mean_loss = np.mean(losses, axis=0)
    std_loss = np.std(losses, axis=0)
    epochs = np.arange(len(mean_loss))
    ax.plot(epochs, mean_loss, label=f"$\\mu B = {mb_size}$", zorder=2)
    ax.fill_between(epochs, mean_loss - std_loss, mean_loss + std_loss, alpha=0.3, zorder=1)

ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Loss')
ax.set_yscale('log')
plt.legend(ncol=2,loc='upper right',frameon=True,framealpha=1, fontsize=18)
ax.set_title(f"1D Regression ($k = {K_SEL}$, $\\eta = {LR}$, rtol = ${lr_labels[RTOL]}$)")
plt.ylim([None,2])
plt.grid(axis='both', linestyle='--', alpha=0.7, zorder=0)
plt.tight_layout()
plt.savefig(PLOT_DIR / "toy_1d_val_loss_microbatchScan.pdf",bbox_inches='tight')

# Polynomial Regression

In [ ]:
colors = sns.color_palette("deep")

fig, ax = plt.subplots(figsize=(8, 6))

K_SEL = 32
RTOL = 1e-2
LR = 0.5

df = datasets['Polynomial']

for mb_size in mb_sizes_polynomial:
    losses = []
    for mseed in model_seeds_polynomial:
        cut = (df['microbatch_size'] == mb_size) & (df['model_seed'] == mseed) & (df['k'] == K_SEL) & (df['rtol'] == RTOL) & (df['lr'] == LR)
        row = df.loc[cut]
        if len(row) > 1:
            print("bad!")
        row = row.iloc[0]
        losses.append(np.array(get_loss_curve(row, 'val')).reshape(1,-1))
    losses = np.concatenate(losses, axis=0)
    mean_loss = np.mean(losses, axis=0)
    std_loss = np.std(losses, axis=0)
    epochs = np.arange(len(mean_loss))
    ax.plot(epochs, mean_loss, label=f"$\\mu B = {mb_size}$", zorder=2)
    ax.fill_between(epochs, mean_loss - std_loss, mean_loss + std_loss, alpha=0.3, zorder=1)

ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Loss')
ax.set_yscale('log')
plt.legend(ncol=2,loc='upper right',frameon=True,framealpha=1, fontsize=18)
ax.set_title(f"Polynomial ($k = {K_SEL}$, $\\eta = {LR}$, rtol = ${lr_labels[RTOL]}$)")
plt.ylim([None,1])
plt.grid(axis='both', linestyle='--', alpha=0.7, zorder=0)
plt.tight_layout()
plt.savefig(PLOT_DIR / "polynomial_val_loss_microbatchScan.pdf",bbox_inches='tight')